# Tutorial 4: Zernikes

`Zernikes` holds Zernike polynomial coefficients — the standard representation
of wavefront errors at one or many field points.

## Key concepts

- Coefficients are indexed by **Noll index** *j* (0 … jmax).
- The 0 index is meaningless.  1 is piston, 4 is defocus, and so on.
- Frame conversions (OCS ↔ CCS) rotate the coefficients using the Zernike
  rotation matrix.
- `to_galsim()` converts to a `galsim.zernike.Zernike` for evaluation and
  visualization.
- `.double()` fits field-dependent coefficients to produce a `DoubleZernikes`
  object.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import Angle
import galsim

from StarSharp import FieldCoords, Zernikes

%matplotlib inline

## 1. Construction

`Zernikes` is a frozen dataclass.  The coefficient array has shape
`(..., nfield, jmax+1)` where the last axis indexes Noll *j*.

In [ ]:
rtp = Angle(30, unit=u.deg)

# Pupil geometry (Rubin-like)
R_outer = 4.18 * u.m
R_inner = 2.56 * u.m

# Create field coords for 4 field points
field = FieldCoords(
    x=np.array([0.0, 0.5, -0.7, 1.2]) * u.deg,
    y=np.array([0.3, -0.4, 0.8, -0.1]) * u.deg,
    frame="ocs",
    rtp=rtp,
)


# Random Zernike coefficients: (4 field points, 29 Noll indices: j=0..28)
rng = np.random.default_rng(42)
jmax = 28
coefs = rng.normal(0, 0.05, (4, jmax + 1)) * u.um
# Zero out piston/tip/tilt for realism
coefs[:, :4] = 0.0 * u.um

zk = Zernikes(
    coefs=coefs,
    field=field,
    R_outer=R_outer,
    R_inner=R_inner,
    wavelength=622.0 * u.nm,
    frame="ocs",
    rtp=rtp,
)

print(f"jmax:   {zk.jmax}")
print(f"nfield: {zk.nfield}")
print(f"eps:    {zk.eps:.3f}")
print(f"frame:  {zk.frame}")
print(f"coefs shape: {zk.coefs.shape}")

## 2. Frame conversions (OCS ↔ CCS)

Zernike polynomials transform under rotation via a **rotation matrix** that mixes
coefficients of the same radial order.  StarSharp uses `galsim.zernike.zernikeRotMatrix`
under the hood.

In [ ]:
zk_ccs = zk.ccs
print(f"OCS → CCS")
print(f"  OCS coefs[0, 4:7] = {zk.coefs[0, 4:7]}")
print(f"  CCS coefs[0, 4:7] = {zk_ccs.coefs[0, 4:7]}")

In [ ]:
# Roundtrip: OCS → CCS → OCS
zk_rt = zk.ccs.ocs
print(f"Roundtrip close: {np.allclose(zk.coefs, zk_rt.coefs)}")
print(f"Max error: {np.max(np.abs(zk.coefs - zk_rt.coefs)):.2e}")

In [ ]:
# Spherical (rotationally symmetric) Zernikes are invariant under rotation
# Noll j=4 is defocus, j=11 is spherical
coefs_symmetric = np.zeros((1, 12)) * u.um
coefs_symmetric[0, 4] = 0.1 * u.um   # defocus
coefs_symmetric[0, 11] = 0.02 * u.um  # spherical

zk_sym = Zernikes(
    coefs=coefs_symmetric, field=field[:1],
    R_outer=R_outer, R_inner=R_inner, frame="ocs", rtp=rtp,
)
zk_sym_ccs = zk_sym.ccs

print("Symmetric Zernikes under rotation:")
print(f"  OCS j=4: {zk_sym.coefs[0, 4]:.4f}  →  CCS j=4: {zk_sym_ccs.coefs[0, 4]:.4f}")
print(f"  OCS j=11: {zk_sym.coefs[0, 11]:.4f}  →  CCS j=11: {zk_sym_ccs.coefs[0, 11]:.4f}")

## 3. GalSim interop

`to_galsim()` converts to a `galsim.zernike.Zernike` object for evaluation
on a pupil grid.  This is useful for wavefront visualization.

In [ ]:
# Convert field point 0 to GalSim Zernike
gz = zk.to_galsim(idx=0, unit=u.um, radius_unit=u.m)
print(f"Type: {type(gz)}")
print(f"R_outer: {gz.R_outer}, R_inner: {gz.R_inner}")

In [ ]:
# Evaluate on a pupil grid and plot the wavefront
npix = 256
x = np.linspace(-R_outer.to_value(u.m), R_outer.to_value(u.m), npix)
xx, yy = np.meshgrid(x, x)
rr = np.sqrt(xx**2 + yy**2)

# Evaluate the GalSim Zernike on the grid
wf = gz.evalCartesian(xx, yy)

# Mask outside the annular pupil
mask = (rr > R_outer.to_value(u.m)) | (rr < R_inner.to_value(u.m))
wf[mask] = np.nan

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    wf, extent=[x[0], x[-1], x[0], x[-1]],
    cmap="RdBu", origin="lower",
)
ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_title("Wavefront at field point 0 [µm]")
plt.colorbar(im, ax=ax, label="WFE [µm]")
ax.set_aspect("equal")

## 4. Bridge: Zernikes → DoubleZernikes

The `.double(kmax, field_outer, field_inner)` method fits the field-dependent
Zernike coefficients to a **DoubleZernike** basis — a product of field Zernikes
(index *k*) and pupil Zernikes (index *j*).

This is useful for compactly representing field-dependent wavefronts.

In [ ]:
# Fit to DoubleZernikes with field kmax=4.
# Should be lossless since kmax=4 matches nfield=4
dz = zk.double(
    kmax=4,
    field_outer=1.75 * u.deg,
    field_inner=0.0 * u.deg,
)

print(f"Type: {type(dz).__name__}")
print(f"kmax (field): {dz.kmax}")
print(f"jmax (pupil): {dz.jmax}")
print(f"coefs shape:  {dz.coefs.shape}")
print(f"frame: {dz.frame}")

In [ ]:
# Go back: DoubleZernikes → Zernikes at the original field points
zk_recovered = dz.single(field)

print(f"Recovered type: {type(zk_recovered).__name__}")
print(f"Recovered nfield: {zk_recovered.nfield}")

# Limited kmax means the roundtrip is approximate
residual = np.max(np.abs(zk.coefs - zk_recovered.coefs))
print(f"Max residual: {residual:.4f}")
print("(Residual depends on how well the field Zernike basis captures the variation)")

In [ ]:
# If we have more field points than kmax, then the fit is lossy
dz_lossy = zk.double(
    kmax=1,
    field_outer=1.75 * u.deg,
    field_inner=0.0 * u.deg,
)
zk_recovered_lossy = dz_lossy.single(field)
residual = np.max(np.abs(zk.coefs - zk_recovered_lossy.coefs))
print(f"Max residual: {residual:.4f}")

## 5. Indexing

In [ ]:
# Integer indexing selects a field point
zk0 = zk[0]
print(f"zk[0]: nfield={zk0.nfield}, jmax={zk0.jmax}")

# Slice indexing
zk_slice = zk[:2]
print(f"zk[:2]: nfield={zk_slice.nfield}")

# Batch dimensions
print(f"batch_shape: {zk.batch_shape}")

## 6. Summary

| Property / Method | Description |
|---|---|
| `.coefs` | Coefficient array, shape `(..., nfield, jmax+1)` |
| `.jmax`, `.nfield`, `.eps` | Derived properties |
| `.ocs`, `.ccs` | Frame conversion (Zernike rotation matrix) |
| `.to_galsim(idx)` | Convert to `galsim.zernike.Zernike` |
| `.double(kmax, field_outer)` | **Bridge to DoubleZernikes** |
| `R_outer`, `R_inner` | Pupil geometry |
| `.field` | Associated `FieldCoords` |

**Next:** [05_DoubleZernikes.ipynb](05_DoubleZernikes.ipynb) — field-dependent
wavefront representation.